# 🟣 Day 15 — Production Skills

**LangGraph from Zero to Advanced · Day 15 of 16**

**Today's goal:** make graphs **reliable, cheap and deployable**.

| Lesson | Topic |
|---|---|
| 1 | Retries for flaky steps (`RetryPolicy`) |
| 2 | Caching expensive steps (`CachePolicy`) |
| 3 | Graceful errors & fallbacks |
| 4 | The Functional API (`@entrypoint`, `@task`) |
| 5 | Testing your graph |
| 6 | Tracing with LangSmith (optional) |
| 7 | Deploying: graph in a `.py` file + `langgraph.json` |
| 8 | Production checklist |

In [ ]:
%pip install -q -U langgraph langchain-openai python-dotenv

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
from langchain_openai import ChatOpenAI
from IPython.display import Image, display

load_dotenv(find_dotenv(usecwd=True))
assert os.getenv("OPENAI_API_KEY"), "Add OPENAI_API_KEY to the .env file in the repo root (see Day 4)."
MODEL = os.getenv("OPENAI_MODEL", "gpt-4.1-nano")
llm = ChatOpenAI(model=MODEL, temperature=0)

def show_graph(app):
    """Draw the graph as a picture, or print it as text when offline."""
    try:
        display(Image(app.get_graph().draw_mermaid_png()))
    except Exception:
        print(app.get_graph().draw_mermaid())

print("✅ Ready")

## Lesson 1 — Retries for flaky steps

APIs fail sometimes (timeouts, rate limits). Add a **`RetryPolicy`** to a node and LangGraph re-runs it automatically, waiting a little longer each time.

We simulate an API that fails twice, then works.

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.types import RetryPolicy

calls = {"count": 0}

class WeatherState(TypedDict):
    city: str
    weather: str

def flaky_weather_api(state: WeatherState) -> dict:
    calls["count"] += 1
    print(f"  attempt {calls['count']}")
    if calls["count"] < 3:
        raise ConnectionError("API timeout")       # fails on attempts 1 and 2
    return {"weather": f"{state['city']}: 29°C, clear"}

g = StateGraph(WeatherState)
g.add_node("weather", flaky_weather_api,
           retry_policy=RetryPolicy(max_attempts=4, initial_interval=0.2, retry_on=ConnectionError))
g.add_edge(START, "weather")
g.add_edge("weather", END)
weather_app = g.compile()

print(weather_app.invoke({"city": "Nashik"}))
assert calls["count"] == 3
print("✅ Succeeded after retries")

| `RetryPolicy` option | Meaning |
|---|---|
| `max_attempts` | total tries (default 3) |
| `initial_interval` | seconds before the first retry |
| `backoff_factor` | wait multiplier each retry (default 2) |
| `retry_on` | which exceptions to retry (don't retry bugs like `KeyError`!) |

💡 `ChatOpenAI` also has its own `max_retries=` for API errors.

## Lesson 2 — Caching expensive steps

If the same input comes again, reuse the old result instead of paying for another LLM call. Add a **`CachePolicy`** to the node and a cache to `compile()`.

In [ ]:
import time
from langgraph.types import CachePolicy
from langgraph.cache.memory import InMemoryCache

class DefineState(TypedDict):
    word: str
    meaning: str

def define(state: DefineState) -> dict:
    return {"meaning": llm.invoke(f"Define '{state['word']}' in 10 words.").content}

g = StateGraph(DefineState)
g.add_node("define", define, cache_policy=CachePolicy(ttl=600))     # cache for 10 minutes
g.add_edge(START, "define")
g.add_edge("define", END)
dict_app = g.compile(cache=InMemoryCache())

for attempt in (1, 2):
    t = time.time()
    out = dict_app.invoke({"word": "serendipity"})
    print(f"call {attempt}: {time.time() - t:.2f}s → {out['meaning']}")

The second call is almost instant — no API call, no cost. The cache key is the node's **input**, so a different word calls the LLM again.

## Lesson 3 — Graceful errors & fallbacks

Don't let one failure crash the whole app. Catch errors **inside the node**, record them in the state, and route to a **fallback**.

```
START ► primary ─ ok ────► END
           └─ error ► fallback ► END
```

In [ ]:
from typing import Literal

class SafeState(TypedDict):
    question: str
    answer: str
    error: str

def primary(state: SafeState) -> dict:
    try:
        if "crash" in state["question"]:
            raise RuntimeError("primary model unavailable")      # simulate an outage
        return {"answer": llm.invoke(state["question"]).content, "error": ""}
    except Exception as e:
        return {"error": str(e)}

def fallback(state: SafeState) -> dict:
    return {"answer": "Sorry, I'm having trouble right now. Please try again in a minute. 🙏"}

def ok_or_fallback(state: SafeState) -> Literal["fallback", "__end__"]:
    return "fallback" if state.get("error") else END

g = StateGraph(SafeState)
g.add_node("primary", primary)
g.add_node("fallback", fallback)
g.add_edge(START, "primary")
g.add_conditional_edges("primary", ok_or_fallback)
g.add_edge("fallback", END)
safe_app = g.compile()

print(safe_app.invoke({"question": "Say hello in Kannada"}))
print(safe_app.invoke({"question": "please crash"}))

💡 LangChain models also support `.with_fallbacks([...])`, e.g. try a nano model, fall back to a bigger one:

In [ ]:
robust_llm = ChatOpenAI(model=MODEL, max_retries=2).with_fallbacks([ChatOpenAI(model="gpt-4.1-mini")])
print(robust_llm.invoke("Reply with just: OK").content)

## Lesson 4 — The Functional API

Not every workflow needs nodes and edges. The **Functional API** lets you write normal Python (`if`, `for`, function calls) and still get checkpointing, streaming and `interrupt()`.

* **`@task`** — a unit of work (like a node). Call it → get a *future* → `.result()`.
* **`@entrypoint`** — the workflow function (like a compiled graph).

In [ ]:
from langgraph.func import entrypoint, task
from langgraph.checkpoint.memory import InMemorySaver

@task
def write_outline(topic: str) -> str:
    return llm.invoke(f"3 bullet outline for a blog on {topic}. Very short.").content

@task
def write_intro(topic: str, outline: str) -> str:
    return llm.invoke(f"Write a 2-sentence intro for a blog on {topic} following:\n{outline}").content

@entrypoint(checkpointer=InMemorySaver())
def blog_workflow(topic: str) -> dict:
    outline = write_outline(topic).result()
    if len(outline) < 10:                       # plain Python logic!
        outline = write_outline(topic + " (detailed)").result()
    intro = write_intro(topic, outline).result()
    return {"outline": outline, "intro": intro}

out = blog_workflow.invoke("monsoon trekking near Pune", {"configurable": {"thread_id": "blog-1"}})
print(out["outline"], "\n\n", out["intro"])

Tasks started together run **in parallel**:

In [ ]:
@task
def fact(topic: str) -> str:
    return llm.invoke(f"One fun fact about {topic}, max 15 words.").content

@entrypoint()
def facts(topics: list[str]) -> list[str]:
    futures = [fact(t) for t in topics]          # start all at once
    return [f.result() for f in futures]         # then wait for all

for line in facts.invoke(["tigers", "the Moon", "mangoes"]):
    print("•", line)

| Graph API | Functional API |
|---|---|
| visual, explicit structure | plain Python control flow |
| best for agents, branches, multi-agent | best for simple scripts, adding LangGraph to existing code |

Both use the same engine — pick what reads best.

## Lesson 5 — Testing your graph

Treat graphs like any code:

1. **Unit-test the logic** — routers and plain-Python nodes are just functions.
2. **Test the graph path** — check which nodes ran with `stream_mode="updates"`.
3. **Smoke-test the LLM nodes** — assert the *shape* of the output (type, not empty), not exact words.

In [ ]:
# 1) unit test a router — no graph, no LLM
assert ok_or_fallback({"error": "boom"}) == "fallback"
assert ok_or_fallback({"error": ""}) == END

# 2) path test — which nodes ran?
def nodes_visited(app, inp):
    return [list(step)[0] for step in app.stream(inp, stream_mode="updates")]

assert nodes_visited(safe_app, {"question": "please crash"}) == ["primary", "fallback"]

# 3) smoke test an LLM node
res = safe_app.invoke({"question": "Name one planet."})
assert isinstance(res["answer"], str) and res["answer"].strip()
print("✅ All tests pass")

💡 Put these in a `tests/test_graph.py` file and run `pytest` before every deploy.

## Lesson 6 — Tracing with LangSmith (optional)

**LangSmith** records every run: each node, each LLM prompt/response, tokens, cost and time. It's the easiest way to debug agents. Free tier available at <https://smith.langchain.com>.

Add these to your `.env` — **no code changes** needed:

```
LANGSMITH_TRACING=true
LANGSMITH_API_KEY=lsv2_...
LANGSMITH_PROJECT=langgraph-course
```

In [ ]:
print("LangSmith tracing is", "ON ✅" if os.getenv("LANGSMITH_TRACING") == "true" else "OFF (optional)")

## Lesson 7 — Deploying

Notebooks are for learning. For an app, move the graph into a **`.py` file** that exposes the compiled graph. Let's write one:

In [ ]:
%%writefile day15_agent.py
"""A small production-style agent module."""
import os
from dotenv import load_dotenv, find_dotenv
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.types import RetryPolicy

load_dotenv(find_dotenv(usecwd=True))
llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL", "gpt-4.1-nano"), temperature=0, max_retries=2)


@tool
def word_count(text: str) -> int:
    """Count the words in a text."""
    return len(text.split())


tools = [word_count]
llm_with_tools = llm.bind_tools(tools)


def agent(state: MessagesState) -> dict:
    return {"messages": [llm_with_tools.invoke(state["messages"])]}


builder = StateGraph(MessagesState)
builder.add_node("agent", agent, retry_policy=RetryPolicy(max_attempts=3))
builder.add_node("tools", ToolNode(tools, handle_tool_errors=True))
builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", tools_condition)
builder.add_edge("tools", "agent")

graph = builder.compile()          # the deploy server adds its own persistence

In [ ]:
from day15_agent import graph

print(graph.invoke({"messages": [("user", "How many words in 'deploying graphs is easy'?")]})["messages"][-1].content)

### Option A — LangGraph server (`langgraph dev`)

Create `langgraph.json` next to the file:

```json
{
  "dependencies": ["."],
  "graphs": { "agent": "./day15_agent.py:graph" },
  "env": ".env"
}
```

Then in a terminal:

```bash
pip install -U "langgraph-cli[inmem]"
langgraph dev
```

You get a local API server with threads, memory, streaming, and **LangGraph Studio** (a visual debugger) in the browser.

### Option B — your own FastAPI app

```python
from fastapi import FastAPI
from day15_agent import graph

app = FastAPI()

@app.post("/chat")
async def chat(message: str):
    result = await graph.ainvoke({"messages": [("user", message)]})
    return {"reply": result["messages"][-1].content}
```

Run with `uvicorn main:app --reload`. (Add a checkpointer like `PostgresSaver` for memory.)

## Lesson 8 — Production checklist ✅

| Area | Do this |
|---|---|
| Secrets | keys in `.env` / secret manager, never in code |
| Loops | `recursion_limit` + your own step counter |
| Failures | `RetryPolicy`, `max_retries`, try/except + fallback node |
| Cost | cache repeated calls, trim/summarize history, small models (nano) for easy steps |
| Memory | `PostgresSaver` / `PostgresStore` instead of in-memory |
| Safety | `interrupt()` before risky actions (payments, emails, deletes) |
| Quality | structured output for decisions; evaluator loops for important text |
| Visibility | LangSmith tracing; stream progress to users |
| Tests | unit-test routers & tools, path tests, LLM smoke tests |

## 🎯 Day 15 Recap

* `RetryPolicy` for flaky steps, `CachePolicy` + `InMemoryCache` to avoid repeat costs.
* Catch errors in nodes and route to fallbacks; `.with_fallbacks()` for models.
* Functional API (`@entrypoint`, `@task`) = LangGraph power with plain Python.
* Test routers, paths, and LLM output shape.
* Deploy: graph in a `.py` file → `langgraph.json` + `langgraph dev`, or FastAPI.

**Tomorrow (Day 16):** 🏆 **Capstone** — build a complete AI research assistant using everything you learned.